# Day 16 · PM Take-Home Assignment
## EDA Report: Retail Sales Dataset
**PG Diploma · AI-ML & Agentic AI Engineering · IIT Gandhinagar**

---
- **Dataset:** `eda_assignment_data.csv` — 1200 rows × 10 columns (retail transactions)
- **Columns:** `category`, `region`, `customer_age`, `unit_price`, `quantity`, `discount_pct`, `revenue`, `customer_rating`, `days_to_ship`, `is_returned`
---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── Global style ──────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

df = pd.read_csv('eda_assignment_data.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()
print('\nNull counts:')
print(df.isnull().sum())
print('\nDescriptive stats:')
df.describe().round(2)

---
## Part A — Comprehensive EDA Report
### A1. Distribution Plots: Histogram + KDE (3 numerical features)

In [ ]:
# ── A1-1: customer_age distribution ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df['customer_age'], bins=25, kde=True, color='steelblue',
             edgecolor='white', line_kws={'lw': 2}, ax=ax)
ax.set_title('Distribution of Customer Age', fontsize=14, fontweight='bold')
ax.set_xlabel('Customer Age (years)')
ax.set_ylabel('Count')
ax.axvline(df['customer_age'].mean(), color='crimson', ls='--', lw=1.5, label=f'Mean = {df["customer_age"].mean():.1f}')
ax.legend()
plt.tight_layout()
plt.savefig('dist_customer_age.png', dpi=150)
plt.show()

**Insight 1:** Customer age follows a near-normal distribution centred around 38 years, suggesting a predominantly working-age adult customer base. The slight right tail implies a smaller but non-trivial older demographic worth targeting with specialised offers.

In [ ]:
# ── A1-2: unit_price distribution ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df['unit_price'], bins=30, kde=True, color='darkorange',
             edgecolor='white', line_kws={'lw': 2}, ax=ax)
ax.set_title('Distribution of Unit Price', fontsize=14, fontweight='bold')
ax.set_xlabel('Unit Price (₹ / $)')
ax.set_ylabel('Count')
ax.axvline(df['unit_price'].median(), color='navy', ls='--', lw=1.5, label=f'Median = {df["unit_price"].median():.1f}')
ax.legend()
plt.tight_layout()
plt.savefig('dist_unit_price.png', dpi=150)
plt.show()

**Insight 2:** Unit price is strongly right-skewed, with the bulk of transactions below ₹200 but a long tail extending past ₹500 (likely high-end electronics). This skew means the mean (≈₹111) misleads as a central measure — the median (≈₹76) better reflects the typical transaction.

In [ ]:
# ── A1-3: revenue distribution ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df['revenue'], bins=40, kde=True, color='seagreen',
             edgecolor='white', line_kws={'lw': 2}, ax=ax)
ax.set_title('Distribution of Revenue per Transaction', fontsize=14, fontweight='bold')
ax.set_xlabel('Revenue')
ax.set_ylabel('Count')
median_rev = df['revenue'].median()
ax.axvline(median_rev, color='darkred', ls='--', lw=1.5, label=f'Median = {median_rev:.0f}')
ax.legend()
plt.tight_layout()
plt.savefig('dist_revenue.png', dpi=150)
plt.show()

**Insight 3:** Revenue is heavily right-skewed due to both the price skew and variable quantities. Most transactions generate under ₹400, but a small number of high-value electronics purchases significantly inflate the mean. Log-transforming revenue would be advisable before any regression modelling.

---
### A2. Relationship Plots: Scatter with Regression Line (sns.regplot)

In [ ]:
# ── A2-1: unit_price vs revenue ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
sns.regplot(data=df, x='unit_price', y='revenue',
            scatter_kws={'alpha': 0.3, 's': 20, 'color': 'steelblue'},
            line_kws={'color': 'crimson', 'lw': 2}, ax=ax)
ax.set_title('Unit Price vs Revenue  (with Regression Line)', fontsize=14, fontweight='bold')
ax.set_xlabel('Unit Price')
ax.set_ylabel('Revenue')
r, p = stats.pearsonr(df['unit_price'], df['revenue'])
ax.annotate(f'r = {r:.3f}  (p < 0.001)', xy=(0.65, 0.05), xycoords='axes fraction',
            fontsize=10, color='crimson',
            bbox=dict(boxstyle='round,pad=0.3', fc='lightyellow', ec='crimson'))
plt.tight_layout()
plt.savefig('scatter_price_revenue.png', dpi=150)
plt.show()

**Insight:** There is a strong positive linear relationship (r ≈ 0.8) between unit price and revenue, confirming that higher-priced items drive total transaction value. The fan-shaped variance (heteroscedasticity) at higher prices suggests quantity variance among expensive items.

In [ ]:
# ── A2-2: customer_age vs customer_rating ─────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
sns.regplot(data=df, x='customer_age', y='customer_rating',
            scatter_kws={'alpha': 0.25, 's': 20, 'color': 'mediumpurple'},
            line_kws={'color': 'darkorange', 'lw': 2}, ax=ax)
ax.set_title('Customer Age vs Rating  (with Regression Line)', fontsize=14, fontweight='bold')
ax.set_xlabel('Customer Age (years)')
ax.set_ylabel('Customer Rating (1–5)')
r2, p2 = stats.pearsonr(df['customer_age'], df['customer_rating'])
ax.annotate(f'r = {r2:.3f}', xy=(0.05, 0.9), xycoords='axes fraction',
            fontsize=10, color='darkorange',
            bbox=dict(boxstyle='round,pad=0.3', fc='lightyellow', ec='darkorange'))
plt.tight_layout()
plt.savefig('scatter_age_rating.png', dpi=150)
plt.show()

**Insight:** Customer age shows negligible correlation with rating (r ≈ 0.0), indicating that satisfaction is not age-dependent. The near-flat regression line tells us age should not be a primary feature in any rating-prediction model.

---
### A3. Correlation Heatmap — All Numerical Features

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)   # show lower triangle only
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, linecolor='white',
            square=True, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Heatmap — Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150)
plt.show()

**Insight:** Revenue is most strongly correlated with `unit_price` (r ≈ 0.80) and `quantity` (r ≈ 0.40). `discount_pct` has a mild negative correlation with revenue, confirming discounting erodes transaction value. Most other pairs are near-zero, indicating low multicollinearity among predictors.

---
### A4. Box Plot — Key Metric Across All Categories

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
order = df.groupby('category')['revenue'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='category', y='revenue', order=order,
            palette='Set2', width=0.55, flierprops={'marker':'o','markersize':3,'alpha':0.4},
            ax=ax)
ax.set_title('Revenue Distribution by Product Category', fontsize=14, fontweight='bold')
ax.set_xlabel('Product Category')
ax.set_ylabel('Revenue per Transaction')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig('boxplot_revenue_category.png', dpi=150)
plt.show()

**Insight:** Electronics transactions have both the highest median revenue and the widest IQR, reflecting the diverse price range from budget to premium devices. Food & Clothing show tightly clustered, lower-revenue distributions with fewer outliers, indicating more consistent, lower-value transactions.

---
## Part B — Stretch Problem: Interactive Plotly Charts

> **Note:** Requires `pip install plotly`. Run `pip install plotly` in your terminal first.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# ── B1: Interactive Histogram — Unit Price ────────────────────────────────
fig_b1 = px.histogram(
    df, x='unit_price', nbins=40, marginal='rug',
    color_discrete_sequence=['steelblue'],
    title='Interactive Distribution of Unit Price',
    labels={'unit_price': 'Unit Price'},
    template='plotly_white'
)
fig_b1.update_layout(bargap=0.05)
fig_b1.write_html('plotly_hist_unit_price.html')
fig_b1.show()

In [ ]:
# ── B2: Interactive Scatter — unit_price vs revenue, coloured by category ─
fig_b2 = px.scatter(
    df, x='unit_price', y='revenue', color='category',
    hover_data=['quantity', 'discount_pct', 'customer_rating', 'region'],
    trendline='ols',
    title='Unit Price vs Revenue by Category (Interactive)',
    labels={'unit_price': 'Unit Price', 'revenue': 'Revenue'},
    opacity=0.6,
    template='plotly_white'
)
fig_b2.write_html('plotly_scatter_price_revenue.html')
fig_b2.show()

In [ ]:
# ── B3: Interactive Box Plot — Revenue by Category ────────────────────────
fig_b3 = px.box(
    df, x='category', y='revenue', color='category',
    points='outliers', hover_data=['region', 'unit_price'],
    title='Revenue by Category (Interactive Box Plot)',
    labels={'revenue': 'Revenue', 'category': 'Category'},
    template='plotly_white'
)
fig_b3.write_html('plotly_box_revenue_category.html')
fig_b3.show()

### B4. Insights Easier to See in Interactive vs Static Charts

**Insight Type 1 — Outlier Identification:**  
In the interactive scatter plot (B2), hovering over outlier points immediately reveals their `category`, `quantity`, `discount_pct`, and `region`. In the static version, we can only observe *that* outliers exist — not *which* specific transactions they are or what makes them extreme. For fraud detection or quality control, this hover context is indispensable.

**Insight Type 2 — Segment-Level Pattern Discovery:**  
Interactive charts support toggling categories on/off via the legend. In B2, clicking 'Electronics' off instantly reveals that the positive price-revenue trend holds across *all* categories — not just the high-price electronics. In a static chart, overlapping point clouds from all 5 categories make it nearly impossible to isolate each segment's slope, leading to Simpson's Paradox-type misreadings.

---
## Part C — Interview Ready

### C1. Violin Plot vs Box Plot

Use a **violin plot** instead of a box plot when you want to show the *full shape of the distribution*, not just summary statistics.  

A box plot reveals: median, IQR (Q1–Q3), whiskers (1.5×IQR), and individual outliers. It is concise and easy to compare across many groups, but it **hides** the underlying distribution shape — you cannot tell from a box plot whether data is unimodal, bimodal, or skewed in a specific direction.

A violin plot wraps a kernel density estimate (KDE) on both sides of the box, showing **probability density at every value**. This makes it ideal when:  
- The distribution is suspected to be bimodal (e.g., two customer age clusters).  
- You care about where the mass is concentrated within the IQR.  
- Comparing the *shape* of distributions across groups matters (e.g., does Electronics have a bimodal price distribution?).  

**Trade-off:** Violin plots are harder for non-technical audiences to read and become misleading with small samples (< ~30), where the KDE is unreliable.

In [ ]:
# Demonstration: Box vs Violin for revenue by category
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

sns.boxplot(data=df, x='category', y='revenue', palette='Set2', ax=axes[0])
axes[0].set_title('Box Plot — Revenue by Category', fontweight='bold')
axes[0].set_xlabel('Category'); axes[0].set_ylabel('Revenue')
axes[0].tick_params(axis='x', rotation=15)

sns.violinplot(data=df, x='category', y='revenue', palette='Set2',
               inner='box', cut=0, ax=axes[1])
axes[1].set_title('Violin Plot — Same Data (Shape Revealed)', fontweight='bold')
axes[1].set_xlabel('Category'); axes[1].set_ylabel('')
axes[1].tick_params(axis='x', rotation=15)

plt.suptitle('Box vs Violin: Electronics shows a notably different density shape',
             fontsize=11, y=1.01, style='italic')
plt.tight_layout()
plt.savefig('box_vs_violin.png', dpi=150)
plt.show()

### C2. `plot_numerical_eda()` Function — Matplotlib OO API

In [ ]:
from scipy import stats

def plot_numerical_eda(df: pd.DataFrame, save_path: str = 'eda_numerics.png') -> None:
    """
    For every numerical column in df, produce a 1×3 panel:
      - Histogram (with KDE overlay)
      - Box plot
      - QQ plot (scipy.stats.probplot)

    Uses the Matplotlib Object-Oriented API throughout.
    Saves the full multi-panel figure to `save_path`.

    Parameters
    ----------
    df        : pandas DataFrame (any numeric columns processed; categoricals ignored)
    save_path : file path for the saved PNG (default: 'eda_numerics.png')
    """
    num_cols = df.select_dtypes(include=np.number).columns.tolist()
    if not num_cols:
        print('No numerical columns found.')
        return

    n = len(num_cols)
    fig, axes = plt.subplots(n, 3, figsize=(15, 4 * n))

    # Ensure axes is always 2D (handles n=1 edge case)
    if n == 1:
        axes = axes[np.newaxis, :]

    colours = plt.cm.tab10.colors

    for i, col in enumerate(num_cols):
        colour = colours[i % len(colours)]
        series = df[col].dropna()

        # ── Panel 1: Histogram + KDE ───────────────────────────────────────
        ax_hist = axes[i, 0]
        ax_hist.hist(series, bins='auto', color=colour, alpha=0.7,
                     edgecolor='white', density=True)
        kde_x = np.linspace(series.min(), series.max(), 300)
        kde = stats.gaussian_kde(series)
        ax_hist.plot(kde_x, kde(kde_x), color='black', lw=2)
        ax_hist.set_title(f'{col} — Histogram + KDE', fontsize=11, fontweight='bold')
        ax_hist.set_xlabel(col)
        ax_hist.set_ylabel('Density')
        ax_hist.axvline(series.mean(), color='crimson', ls='--', lw=1.2,
                        label=f'μ={series.mean():.2f}')
        ax_hist.legend(fontsize=8)

        # ── Panel 2: Box Plot ──────────────────────────────────────────────
        ax_box = axes[i, 1]
        bp = ax_box.boxplot(series, vert=True, patch_artist=True,
                            medianprops={'color': 'white', 'lw': 2},
                            flierprops={'marker': 'o', 'markersize': 3,
                                        'markerfacecolor': colour, 'alpha': 0.5})
        bp['boxes'][0].set_facecolor(colour)
        bp['boxes'][0].set_alpha(0.7)
        ax_box.set_title(f'{col} — Box Plot', fontsize=11, fontweight='bold')
        ax_box.set_ylabel(col)
        ax_box.set_xticks([])

        q1, q3 = series.quantile(0.25), series.quantile(0.75)
        ax_box.annotate(f'Q1={q1:.1f}\nQ3={q3:.1f}\nIQR={q3-q1:.1f}',
                        xy=(1.35, series.median()), fontsize=7.5,
                        va='center', ha='left',
                        bbox=dict(boxstyle='round,pad=0.2', fc='lightyellow', ec='grey'))

        # ── Panel 3: QQ Plot ───────────────────────────────────────────────
        ax_qq = axes[i, 2]
        (osm, osr), (slope, intercept, r) = stats.probplot(series, dist='norm')
        ax_qq.scatter(osm, osr, color=colour, alpha=0.5, s=15, label='Data quantiles')
        fit_line = np.array(osm) * slope + intercept
        ax_qq.plot(osm, fit_line, 'r-', lw=2, label=f'Normal fit (r={r:.3f})')
        ax_qq.set_title(f'{col} — QQ Plot', fontsize=11, fontweight='bold')
        ax_qq.set_xlabel('Theoretical Quantiles')
        ax_qq.set_ylabel('Sample Quantiles')
        ax_qq.legend(fontsize=8)

    fig.suptitle('Numerical EDA: Histogram · Box · QQ per Feature',
                 fontsize=14, fontweight='bold', y=1.005)
    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f'Saved → {save_path}  ({n} columns × 3 panels)')
    plt.show()


# ── Run it ────────────────────────────────────────────────────────────────
plot_numerical_eda(df)

### C3. Chart Critique

---

#### (a) 3D Pie Chart with 12 Segments for Market Share

**Problem:** Pie charts are already poor for comparing more than 5–6 segments because human perception struggles to judge arc/angle sizes accurately. Adding a 3D perspective makes this drastically worse — segments at the front appear proportionally larger than those at the back (foreshortening distortion). With 12 segments, many slices will share almost identical sizes, making comparison essentially guesswork, even with data labels.

**Better Alternative:** A horizontal bar chart sorted by market share descending. Bars use length along a common baseline — the most perceptually accurate preattentive attribute. 12 categories fit comfortably, percentage labels can be added directly, and ranking is instantly visible.

---

#### (b) Line Chart for Survey Scores Across 5 Unordered Categories

**Problem:** Line charts encode *trend over a continuous or ordered axis* — they imply a sequential relationship between adjacent points. Survey categories like `["Price", "Quality", "Support", "Design", "Speed"]` have no inherent order. Drawing a line between them fabricates a non-existent trend (e.g., it would suggest Quality "transitions" to Support), misleading viewers into looking for a pattern that doesn't exist.

**Better Alternative:** A bar chart (vertical or horizontal). Each category gets a separate bar anchored at a common zero baseline. If comparing multiple respondent groups, use a grouped bar chart. If the goal is showing *deviation from a benchmark*, use a diverging bar chart.

---

#### (c) Scatter Plot with 500k Points and No Transparency

**Problem:** With 500,000 fully opaque points, almost all of them will overlap completely (overplotting). The chart becomes a solid black blob, hiding all density information — ironically, areas with the most data become visually indistinguishable from areas with moderate data. Outliers at the edges are visible but the core structure is entirely obscured.

**Better Alternatives (choose based on goal):**
- **`alpha=0.05` transparency** — quick fix, reveals density through saturation buildup.
- **Hexbin plot (`plt.hexbin`)** — bins points into hexagonal cells coloured by count; excellent for 100k+ points.
- **2D KDE contour plot (`sns.kdeplot`)** — shows smooth density contours, ideal for communicating where data concentrates.
- **Sampled scatter** — randomly sample 2–5k points for the visual; confirm patterns hold on the full dataset separately.

---
## Part D — AI-Augmented Task

### D1. AI Prompt Used

> *"Generate a complete Python EDA script for a retail sales dataset. Include 6 charts: distributions, correlations, time trends, category comparisons, and one unusual/creative chart."*

### D2. Generated & Executed Script

In [ ]:
# ── AI-Generated EDA Script (prompt response — run & evaluated below) ─────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

df_ai = pd.read_csv('eda_assignment_data.csv')
sns.set_theme(style='whitegrid')

fig = plt.figure(figsize=(18, 14))
fig.suptitle('AI-Generated Retail EDA Dashboard', fontsize=16, fontweight='bold', y=1.01)

gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# Chart 1: Revenue distribution
ax1 = fig.add_subplot(gs[0, 0])
sns.histplot(df_ai['revenue'], bins=35, kde=True, color='teal', ax=ax1)
ax1.set_title('Revenue Distribution', fontweight='bold')
ax1.set_xlabel('Revenue'); ax1.set_ylabel('Count')

# Chart 2: Correlation heatmap
ax2 = fig.add_subplot(gs[0, 1:3])
corr_ai = df_ai.select_dtypes(include=np.number).corr()
sns.heatmap(corr_ai, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            linewidths=0.5, ax=ax2, cbar_kws={'shrink': 0.8})
ax2.set_title('Correlation Matrix', fontweight='bold')

# Chart 3: Revenue by category (bar)
ax3 = fig.add_subplot(gs[1, 0])
cat_rev = df_ai.groupby('category')['revenue'].mean().sort_values(ascending=False)
ax3.bar(cat_rev.index, cat_rev.values, color=sns.color_palette('Set2', len(cat_rev)))
ax3.set_title('Avg Revenue by Category', fontweight='bold')
ax3.set_xlabel(''); ax3.set_ylabel('Avg Revenue')
ax3.tick_params(axis='x', rotation=20)

# Chart 4: Days-to-ship trend across discount bins (simulated time/trend)
ax4 = fig.add_subplot(gs[1, 1])
df_ai['discount_bin'] = pd.cut(df_ai['discount_pct'], bins=5)
ship_trend = df_ai.groupby('discount_bin', observed=True)['days_to_ship'].mean()
ax4.plot(range(len(ship_trend)), ship_trend.values, marker='o', color='coral', lw=2)
ax4.set_xticks(range(len(ship_trend)))
ax4.set_xticklabels([str(b) for b in ship_trend.index], rotation=30, fontsize=7)
ax4.set_title('Avg Ship Days vs Discount Bin', fontweight='bold')
ax4.set_xlabel('Discount Bucket'); ax4.set_ylabel('Avg Days to Ship')

# Chart 5: Rating by region (grouped boxplot)
ax5 = fig.add_subplot(gs[1, 2])
sns.boxplot(data=df_ai, x='region', y='customer_rating', palette='pastel', ax=ax5)
ax5.set_title('Rating by Region', fontweight='bold')
ax5.set_xlabel('Region'); ax5.set_ylabel('Customer Rating')

# Chart 6 (UNUSUAL): Radar / Spider chart — avg feature profile per category
# Using polar coordinates — each category gets a polygon over normalised metrics
ax6 = fig.add_subplot(gs[2, :], polar=True)

metrics = ['unit_price', 'quantity', 'discount_pct', 'customer_rating', 'days_to_ship']
categories_list = df_ai['category'].unique()
N = len(metrics)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close polygon

# Normalise each metric 0–1
norm_df = df_ai[metrics].copy()
for m in metrics:
    norm_df[m] = (norm_df[m] - norm_df[m].min()) / (norm_df[m].max() - norm_df[m].min())
norm_df['category'] = df_ai['category'].values
cat_means = norm_df.groupby('category')[metrics].mean()

colours_radar = sns.color_palette('tab10', len(categories_list))
for idx, cat in enumerate(categories_list):
    values = cat_means.loc[cat].tolist()
    values += values[:1]
    ax6.plot(angles, values, lw=2, color=colours_radar[idx], label=cat)
    ax6.fill(angles, values, alpha=0.08, color=colours_radar[idx])

ax6.set_xticks(angles[:-1])
ax6.set_xticklabels(metrics, fontsize=10)
ax6.set_yticks([0.25, 0.5, 0.75])
ax6.set_yticklabels(['25%', '50%', '75%'], fontsize=8)
ax6.set_title('Radar Chart: Normalised Feature Profile by Category',
              fontweight='bold', pad=20)
ax6.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=9)

plt.savefig('ai_eda_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dashboard saved → ai_eda_dashboard.png')

### D3. Evaluation of AI-Generated Script

**Are charts labelled?**  
Yes — every chart has a title, x/y axis labels, and the radar chart includes a legend. The correlation heatmap uses annotated values. One minor gap: the bar chart's x-axis tick labels are rotated 20° which is slightly insufficient for longer category names; 30–45° would be cleaner.

**Does the unusual chart (Radar / Spider) add insight?**  
Yes, meaningfully. The radar chart reveals *multi-dimensional category profiles at a glance*: Electronics shows high normalised unit price but average discount and ship time; Food is low on price/quantity but surprisingly high on rating; Sports scores consistently across most metrics. This cross-feature comparison is impossible to see from individual charts. It is not merely decorative.

**What changes would make it portfolio-quality?**

1. **Consistent colour palette across all 6 charts** — currently each chart uses a different palette (`teal`, `Set2`, `tab10`, etc.), creating a fragmented look. A shared 5-colour palette for the 5 categories would unify the dashboard.
2. **Add a subtitle/caption below the radar chart** explaining what normalisation was applied, as the 0–1 scale is not self-evident to non-technical viewers.
3. **Replace the discount-bin line chart (Chart 4)** — using a categorical x-axis with a line chart has the same problem as C3(b). A bar chart would be more appropriate here.
4. **Add figure-level footnote** with dataset source, row count, and date — essential for any portfolio or presentation artefact.
5. **Export at 300 DPI** for print/presentation quality (currently 150 DPI).

---
## Final Dashboard Export — Publication Quality

In [ ]:
# ── Export all key Part-A charts into a single dashboard PNG ─────────────
fig_dash = plt.figure(figsize=(20, 18))
fig_dash.suptitle(
    'EDA Dashboard — Retail Sales Dataset\n'
    'Day 16 · PM Assignment · IIT Gandhinagar PG Diploma',
    fontsize=15, fontweight='bold', y=1.01
)

gs_d = gridspec.GridSpec(3, 3, figure=fig_dash, hspace=0.50, wspace=0.35)

# Row 0: three distribution plots
for col_idx, (col, clr) in enumerate(zip(
        ['customer_age', 'unit_price', 'revenue'],
        ['steelblue', 'darkorange', 'seagreen'])):
    ax = fig_dash.add_subplot(gs_d[0, col_idx])
    sns.histplot(df[col], bins=25, kde=True, color=clr,
                 edgecolor='white', line_kws={'lw': 2}, ax=ax)
    ax.set_title(f'Distribution: {col}', fontweight='bold')
    ax.set_xlabel(col); ax.set_ylabel('Count')

# Row 1: two regression scatters + correlation heatmap
ax_s1 = fig_dash.add_subplot(gs_d[1, 0])
sns.regplot(data=df, x='unit_price', y='revenue',
            scatter_kws={'alpha': 0.25, 's': 15, 'color': 'steelblue'},
            line_kws={'color': 'crimson', 'lw': 2}, ax=ax_s1)
ax_s1.set_title('Price vs Revenue', fontweight='bold')

ax_s2 = fig_dash.add_subplot(gs_d[1, 1])
sns.regplot(data=df, x='customer_age', y='customer_rating',
            scatter_kws={'alpha': 0.2, 's': 15, 'color': 'mediumpurple'},
            line_kws={'color': 'darkorange', 'lw': 2}, ax=ax_s2)
ax_s2.set_title('Age vs Rating', fontweight='bold')

ax_hm = fig_dash.add_subplot(gs_d[1, 2])
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.4, ax=ax_hm, cbar=False, annot_kws={'size': 7})
ax_hm.set_title('Correlation Heatmap', fontweight='bold')

# Row 2: boxplot spanning full width
ax_box_d = fig_dash.add_subplot(gs_d[2, :])
sns.boxplot(data=df, x='category', y='revenue', order=order,
            palette='Set2', width=0.55,
            flierprops={'marker': 'o', 'markersize': 3, 'alpha': 0.4},
            ax=ax_box_d)
ax_box_d.set_title('Revenue Distribution by Product Category', fontweight='bold')
ax_box_d.set_xlabel('Category'); ax_box_d.set_ylabel('Revenue')

fig_dash.text(0.5, -0.01,
              'Dataset: eda_assignment_data.csv  |  n=1,200  |  Day 16 PM  |  IIT Gandhinagar',
              ha='center', fontsize=9, color='grey', style='italic')

fig_dash.savefig('eda_dashboard.png', dpi=200, bbox_inches='tight')
print('Publication-quality dashboard saved → eda_dashboard.png')
plt.show()